In [ ]:
# Run the Gradio UI (this will block and print the share URL)
import sys, os
repo = '/content/project'
if repo not in sys.path:
    sys.path.insert(0, repo)
print('Starting UI using /content/project/colab_gradio_ui.py')
!python /content/project/colab_gradio_ui.py

In [ ]:
# Install ffmpeg and Python dependencies
!apt-get update -y && apt-get install -y ffmpeg
!pip install -r /content/project/requirements.txt gradio --quiet

In [ ]:
# Clone repo if missing (or pull latest)
import os, subprocess
repo = '/content/project'
if not os.path.exists(repo):
    print('Cloning repo...')
    subprocess.run(['git','clone','https://github.com/hanhwannau/A', repo, '--depth','1'], check=True)
else:
    print('Repo exists, attempting pull...')
    subprocess.run(['git','-C', repo, 'pull'], check=False)

!ls -la /content/project/app/plugins || true

In [ ]:
# Ensure repo is on disk and show key paths
!pwd
!ls -la /content || true
!ls -la /content/project || true
!ls -la /content/project/app/plugins || true

import os, sys
print('project exists:', os.path.exists('/content/project'))
print('runner exists:', os.path.exists('/content/project/app/plugins/runner.py'))
print('sys.path sample:', sys.path[:6])

# One-click Colab video pipeline

Chỉ cần chạy cell bên dưới trong Google Colab.
Nó sẽ:
- mount Google Drive
- clone repo nếu cần
- cài dependencies
- khởi chạy Gradio web UI
- tạo link để bạn điều khiển backend

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    print('Drive mount skipped or already mounted:', exc)

repo_path = Path('/content/project')
if not repo_path.exists():
    candidates = [
        Path('/content/drive/MyDrive/A'),
        Path('/content/drive/My Drive/A'),
        Path('/content/drive/MyDrive/video-pipeline'),
        Path('/content/drive/My Drive/video-pipeline'),
    ]
    for candidate in candidates:
        if candidate.exists():
            repo_path = candidate.resolve()
            break
    else:
        print('Cloning GitHub repo...')
        subprocess.run([
            'git', 'clone', 'https://github.com/hanhwannau/A', str(repo_path)
        ], check=True)
        repo_path = repo_path.resolve()

print('Using repo at:', repo_path)
if not (repo_path / 'app' / 'plugins' / 'runner.py').exists():
    raise FileNotFoundError(f'Repo root invalid, missing app/plugins/runner.py at {repo_path}')
os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', 'gradio'], check=True)

import gradio as gr
from app.plugins.runner import run_from_config


def process(video_file):
    if video_file is None:
        return 'No video uploaded', '', 'Upload a video file first.'

    if isinstance(video_file, (str, Path)):
        video_path = Path(video_file)
    else:
        video_path = Path(getattr(video_file, 'name', str(video_file)))

    if not video_path.exists():
        return 'failed', '', f'Video file not found: {video_path}'

    print('Running pipeline for', video_path)
    result = run_from_config('config_pipeline_full.yaml', video_url=str(video_path))
    rendered_path = Path(result.get('rendered_path', 'outputs/final.mp4'))
    if not rendered_path.is_absolute():
        rendered_path = repo_path / rendered_path

    if rendered_path.exists():
        return 'success', str(rendered_path), 'Rendered output is ready.'

    return 'failed', '', f'Output not found. {result.get("render_error", "")}'

with gr.Blocks() as demo:
    gr.Markdown('# One-click Colab Video Pipeline')
    gr.Markdown('Upload a video and click Run. The backend will execute on Colab.')
    video_input = gr.File(label='Upload video', file_count='single', type='file')
    status = gr.Textbox(label='Status', interactive=False)
    output_path = gr.Textbox(label='Rendered output path', interactive=False)
    message = gr.Textbox(label='Message', interactive=False)
    run_button = gr.Button('Run pipeline')
    run_button.click(process, inputs=[video_input], outputs=[status, output_path, message])


demo.launch(share=True)
